# Session 6: Naive Bayes from Scratch: an SMS Spam Filter

## Overview

A mobile operator wants to filter spam text messages before they reach its customers. In this session you build
the filter from nothing but word counts and **Bayes' rule**: the **naive Bayes** classifier. It is one of the
oldest machine-learning methods for text, still used because it is fast, needs little data, and is easy to
inspect. Then you check that scikit-learn's `MultinomialNB` gives exactly the same answers.

By the end, you will be able to:

- Turn text into word counts with a simple tokeniser.
- Apply Bayes' rule to classification, and state the "naive" independence assumption.
- Estimate word probabilities with Laplace smoothing, and explain why smoothing is needed.
- Work with log-probabilities to avoid numerical underflow.
- Evaluate a spam filter with precision and recall, and inspect which words drive its decisions.
- Reproduce the hand-built model with `CountVectorizer` and `MultinomialNB`.

**Prerequisites:** Session 5 (confusion matrix, precision, recall); conditional probability and Bayes' rule
([6. Statistical Foundations](../6.%20Statistical%20Foundations%20for%20Data%20Science), Session 1).

**Dataset:** [UCI SMS Spam Collection](https://archive.ics.uci.edu/dataset/228/sms+spam+collection). 5,574 real
SMS messages in English, collected for research in 2011-2012, each labelled **ham** (legitimate) or **spam**. The
code downloads the zip file from UCI. Some spam messages contain adult content, as real spam does.

## Where this fits

| Part | Sessions | Task | Algorithms |
| --- | --- | --- | --- |
| 1. Regression | 1, 2, 3, 4 | Predict a number | Linear regression by gradient descent and in scikit-learn, tree ensembles, explaining black-box models |
| 2. Classification | 5, 6, 7, 8, 9 | Predict a category | Logistic regression, naive Bayes, linear text classifiers, comparing classifiers **← you are here** |
| 3. Clustering | 10, 11 | Find groups without labels | k-means, hierarchical clustering |
| 4. Topic modelling | 12 | Find themes in text | LDA, NMF |

**This session's question:** How well can a classifier built only from word counts and Bayes' rule separate spam
from legitimate messages?

## Step 1: Set up the environment

This module has one pinned `requirements.txt` (in the module folder) for all twelve sessions. If you have
not done it yet, create a virtual environment and install it once, from the module folder:

```bash
python3.11 -m venv .venv
source .venv/bin/activate          # Windows: .venv\Scripts\Activate.ps1
pip install -r requirements.txt
jupyter notebook
```

This session uses:

| Package | Why we need it |
| --- | --- |
| numpy | Log-probabilities |
| pandas | Tables |
| scikit-learn | Train/test split, the reference model, metrics |

Run the cell below. It imports what this notebook needs and prints the versions. The numbers quoted in
this notebook come from these exact versions; a different version can shift the last decimal place.

In [ ]:
import io
import re
import urllib.request
import zipfile
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, precision_score, recall_score
from sklearn.model_selection import train_test_split

from importlib.metadata import version

for package in ["numpy", "pandas", "scikit-learn"]:
    print(f"{package:13s} {version(package)}")

## Step 2: Download the messages

This dataset is not available through `ucimlrepo`, so the cell downloads the zip file from UCI and reads the
`SMSSpamCollection` file in it: one message per line, the label, a tab, then the text.

The collection contains **duplicate messages** (the same spam sent to many people, or common replies such as
"Ok"). As in Session 3, a message that appears in both the training and the test set makes the test too easy, so
we keep one copy of each.

In [ ]:
URL = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
with zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(URL).read())) as archive:
    lines = archive.read("SMSSpamCollection").decode("utf-8").splitlines()
sms = pd.DataFrame([line.split("\t", 1) for line in lines if line.strip()], columns=["label", "message"])
print(f"{len(sms):,} messages, {sms['message'].duplicated().sum()} duplicates")
sms = sms.drop_duplicates("message").reset_index(drop=True)
print(f"{len(sms):,} unique messages:", sms["label"].value_counts().to_dict())

train, test = train_test_split(sms, test_size=0.2, stratify=sms["label"], random_state=42)
print(f"train: {len(train):,}, test: {len(test):,}")

**Read the output:** 5,574 messages, of which 403 are duplicates. That leaves 5,171 unique messages, 653 of them
spam (12.6%). We train on 4,136 and hold out 1,035.

The classes are **imbalanced**: a "filter" that lets everything through would be 87% accurate and completely
useless. Accuracy alone will not tell us much.

## Step 3: What does spam look like?

In [ ]:
pd.set_option("display.max_colwidth", 120)
for label in ["ham", "spam"]:
    print(f"--- {label}")
    print(train.loc[train["label"] == label, "message"].sample(4, random_state=2).to_string(index=False))
print()
print("median length in characters:", train.groupby("label")["message"].apply(lambda m: m.str.len().median()).to_dict())

**Read the output:** ham is short, informal chat ("Thanks. Fills me with complete calm and reassurance!"). Spam is
longer (median 148 characters against 53), and full of offers, prizes, phone numbers, prices in pounds and calls to
action ("URGENT!", "won a £800 prize", "Please CALL 08712402902", "reply"). Individual **words** look like strong clues. Naive Bayes uses exactly that.

## Step 4: From text to words

A **tokeniser** splits text into words (*tokens*). Ours lower-cases the text and keeps runs of letters, digits,
apostrophes and currency signs, so "£1000" and "150p" survive as tokens: in spam, they are informative.

In [ ]:
def tokenize(text):
    return re.findall(r"[a-z0-9£$']+", text.lower())


print(tokenize("WINNER!! You have won a £1000 prize. Call 09061701461 to claim, 150p/msg"))

## Step 5: Bayes' rule for classification

We want $P(\text{spam} \mid \text{words})$. Bayes' rule turns it around into things we can count:

$$P(\text{spam} \mid w_1, \dots, w_k) = \frac{P(\text{spam})\;P(w_1, \dots, w_k \mid \text{spam})}{P(w_1, \dots, w_k)}$$

$P(w_1, \dots, w_k \mid \text{spam})$, the probability of this exact sequence of words in a spam message, is
impossible to estimate: almost every message is unique. The **naive** assumption is that, within a class, words
occur **independently** of each other:

$$P(w_1, \dots, w_k \mid \text{spam}) \approx P(w_1 \mid \text{spam}) \times P(w_2 \mid \text{spam}) \times \dots \times P(w_k \mid \text{spam})$$

It is clearly false ("claim" and "prize" go together), but the classifier only has to rank spam above ham, not
get the probabilities exactly right, and for that it works surprisingly well. The denominator is the same for
both classes, so we compare the numerators and pick the larger.

## Step 6: Count words in each class

Training naive Bayes is just counting:

- the **prior** $P(\text{spam})$: the share of training messages that are spam;
- for every word, how often it appears in spam and in ham.

Then $P(w \mid \text{spam})$ = (count of $w$ in spam) / (total words in spam).

In [ ]:
word_counts = {"ham": Counter(), "spam": Counter()}
for label, message in zip(train["label"], train["message"]):
    word_counts[label].update(tokenize(message))

vocabulary = set(word_counts["ham"]) | set(word_counts["spam"])
total_words = {label: sum(counts.values()) for label, counts in word_counts.items()}
prior = train["label"].value_counts(normalize=True).to_dict()

print(f"vocabulary: {len(vocabulary):,} distinct words")
print(f"total words: {total_words}")
print(f"priors: {prior}")
for word in ["call", "prize", "claim", "love", "later"]:
    print(f"{word:6s} ham: {word_counts['ham'][word]:4d}   spam: {word_counts['spam'][word]:4d}")

**Read the output:** 7,891 distinct words. Spam messages contribute 13,127 words, ham 51,854. "call" appears in
both, but far more often per word in spam. "prize" and "claim" appear **only** in spam (65 and 72 times) and never
in ham; "love" is mostly ham, and "later" appears only in ham.

That zero is a problem. Step 7 explains why.

## Step 7: Laplace smoothing

If "prize" never appears in training ham, then $P(\text{prize} \mid \text{ham}) = 0$, and **any** message containing
"prize" gets a ham probability of exactly zero, however ham-like the rest of it is. One unseen word should not
outweigh everything else.

**Laplace (add-one) smoothing** pretends every word in the vocabulary was seen once more than it was:

$$P(w \mid c) = \frac{\text{count}(w, c) + \alpha}{\text{total words in } c + \alpha \cdot |V|}$$

with $\alpha = 1$ and $|V|$ the vocabulary size. No probability is ever zero, and frequent words are barely
affected.

In [ ]:
ALPHA = 1.0
V = len(vocabulary)


def word_prob(word, label):
    return (word_counts[label][word] + ALPHA) / (total_words[label] + ALPHA * V)


for word in ["prize", "call", "later"]:
    print(f"P({word} | spam) = {word_prob(word, 'spam'):.5f}   P({word} | ham) = {word_prob(word, 'ham'):.5f}   "
          f"ratio = {word_prob(word, 'spam') / word_prob(word, 'ham'):.3f}")

**Read the output:** "prize" is now possible in ham, but about 188 times more likely in spam. "call" is 4.5 times
more likely in spam; "later" is about 29 times more likely in ham (ratio 0.035). Each word in a message
multiplies the odds by its ratio.

## Step 8: Work with logarithms

Multiplying many small probabilities produces numbers too small for the computer to store: they **underflow** to
exactly 0. Taking logs turns products into sums, which stay well-behaved:

$$\log P(c) + \sum_{i} \log P(w_i \mid c)$$

In [ ]:
probs = np.full(400, 1e-4)                   # 400 words, each with probability 1e-4
print("product:", np.prod(probs))
print("sum of logs:", np.sum(np.log(probs)))


def class_scores(message):
    words = [w for w in tokenize(message) if w in vocabulary]     # ignore words never seen in training
    return {label: np.log(prior[label]) + sum(np.log(word_prob(w, label)) for w in words)
            for label in ["ham", "spam"]}


def classify(message):
    scores = class_scores(message)
    return max(scores, key=scores.get)


example = "WINNER!! You have won a £1000 prize. Call now to claim"
print(class_scores(example), "->", classify(example))
print(classify("Sorry, I'll call you later. Love you"))

**Read the output:** the product of 400 probabilities of 1e-4 is `0.0`, not because it is zero, but because it is
smaller than the smallest number a 64-bit float can hold. The sum of logs (-3,684) is no problem.

The prize message gets a much higher log-score for spam than for ham, and is classified as spam; the friendly
message is classified as ham. That is the whole classifier.

## Step 9: Evaluate on the test messages

For a spam filter, the two errors are not equal. A **false positive** (a real message sent to the spam folder)
can mean a missed message from a friend or a bank. A **false negative** (spam in the inbox) is an annoyance. So
**precision** for spam matters most: when we say "spam", we must be right.

In [ ]:
predicted = test["message"].map(classify)
tn, fp, fn, tp = confusion_matrix(test["label"], predicted, labels=["ham", "spam"]).ravel()
print(f"ham kept: {tn}   ham wrongly blocked (FP): {fp}")
print(f"spam missed (FN): {fn}   spam caught: {tp}")
print(f"accuracy {np.mean(predicted == test['label']):.3f}   "
      f"precision {precision_score(test['label'], predicted, pos_label='spam'):.3f}   "
      f"recall {recall_score(test['label'], predicted, pos_label='spam'):.3f}")

**Read the output:** of 904 legitimate test messages, **4** are wrongly blocked; of 131 spam messages, 120 are
caught and 11 get through. Precision is 0.968 and recall 0.916; accuracy is 98.6%, compared with 87% for the
"block nothing" filter.

Not bad for a model that is nothing but word counts.

## Step 10: Look inside the model

The **log ratio** $\log P(w \mid \text{spam}) - \log P(w \mid \text{ham})$ says how strongly each word points
to spam (positive) or ham (negative). Unlike most models, naive Bayes can be fully read off. We only list words
seen at least 10 times, so that rare words with extreme ratios do not crowd the list.

In [ ]:
frequent = [w for w in vocabulary if word_counts["ham"][w] + word_counts["spam"][w] >= 10]
log_ratio = pd.Series({w: np.log(word_prob(w, "spam")) - np.log(word_prob(w, "ham")) for w in frequent})
print("strongest spam words:", log_ratio.nlargest(12).round(1).to_dict())
print("strongest ham words: ", log_ratio.nsmallest(12).round(1).to_dict())

**Read the output:** the spam words read like a spam message: "claim", "prize", "won", "150p", "nokia", "www",
"guaranteed", "£1". The ham words are personal and conversational: "he", "she", "later", "i'll", and text-speak
such as "lor" and "da" (common in the Singaporean part of the collection).

Two surprises: "gt" and "lt" are strong ham words. They are leftovers of the HTML codes `&lt;` and `&gt;` that
replaced placeholders such as `<#>` when the messages were collected (you saw `&amp;` in a ham message in Step 3). The model is right to use them, since they
really do separate the classes *in this data*, but they would mean nothing on a new operator's messages. Always
check what a model relies on.

## Step 11: Read the mistakes

In [ ]:
mistakes = test.assign(predicted=predicted).query("label != predicted")
print("Legitimate messages blocked as spam:")
print(mistakes.query("label == 'ham'")["message"].to_string(index=False))
print("\nSpam that got through (first 4):")
print(mistakes.query("label == 'spam'")["message"].head(4).to_string(index=False))

**Read the output:** the blocked ham messages tend to mention a brand or a commercial word ("Nokia", a phone
line, an offer) in a short message where there is little else to go on. The spam that got through avoids the
classic spam vocabulary: it reads like chat or an ordinary event announcement, or hides its key words in
run-together strings our tokeniser does not split (such as a web address glued to the next word). Naive Bayes only sees *which* words are present, not what the
message means.

## Step 12: The same model in scikit-learn

`CountVectorizer` does Steps 4 and 6 (tokenise, build the vocabulary, count) and returns a matrix with one row per
message and one column per word. `MultinomialNB(alpha=1.0)` does Steps 7-8. We give the vectoriser our token
pattern so both see the same words.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

vectorizer = CountVectorizer(token_pattern=r"[a-z0-9£$']+")      # lower-cases by default
counts_train = vectorizer.fit_transform(train["message"])
print(f"document-term matrix: {counts_train.shape[0]:,} messages x {counts_train.shape[1]:,} words, "
      f"{counts_train.nnz:,} non-zero entries")

sk_nb = MultinomialNB(alpha=1.0).fit(counts_train, train["label"])
sk_predicted = sk_nb.predict(vectorizer.transform(test["message"]))
print(f"agreement with our classifier: {np.mean(sk_predicted == predicted):.1%}")

**Read the output:** the matrix has 4,136 rows and 7,891 columns, but only 59,419 non-zero entries: each
message uses a tiny fraction of the vocabulary. scikit-learn stores it as a **sparse matrix**, keeping only the
non-zeros. The scikit-learn model agrees with ours on **100%** of the test messages: same counts, same smoothing,
same rule.

## What this session hands to the next

- **Naive Bayes from first principles:** priors, word likelihoods, the independence assumption, Laplace smoothing,
  log-probabilities.
- **Text as a document-term matrix** of counts, stored sparsely.
- **A baseline spam filter:** precision 0.968, recall 0.916 on held-out messages.
- Session 7 builds a production-style spam filter with scikit-learn pipelines: better text features (TF-IDF),
  stronger linear classifiers, tuning, and saving the model for deployment.

## Common mistakes and fixes

| Mistake | What goes wrong | Fix |
| --- | --- | --- |
| No smoothing (`alpha=0`) | One unseen word sets a class probability to zero | Laplace smoothing, `alpha=1` |
| Multiplying probabilities | Underflow to 0.0 for long messages | Add log-probabilities |
| Building the vocabulary from all messages | Test words leak into training | `fit_transform` on training text, `transform` on test text |
| Keeping duplicate messages | The same message in train and test; optimistic scores | Drop duplicates before splitting |
| Judging a spam filter by accuracy | "Block nothing" scores 87% | Precision and recall for the spam class |
| Trusting every strong word | Artefacts such as "lt"/"gt" do not generalise | Inspect the top words; clean the text if needed |

## Exercises

- [ ] Re-run Steps 9-12 with `ALPHA = 0.1` and `ALPHA = 5`. How do precision and recall change?
- [ ] Remove the prior (use 0.5 for both classes). How many more ham messages are blocked? Why?
- [ ] Classify with a stricter rule: only call a message spam if its spam score beats the ham score by more than 5.
  What happens to precision and recall?
- [ ] Change the tokeniser to split runs of letters from runs of digits, so that "STOP2stop" becomes "stop", "2",
  "stop". Does recall improve?

Next, Session 7 turns this into a production-style spam filter with scikit-learn pipelines, TF-IDF features, model
comparison and tuning.